In [ ]:
from google.colab import files
uploaded = files.upload()

Saving delhi_ward_combined_risk_final.csv to delhi_ward_combined_risk_final.csv


In [ ]:
import threading
import time
import csv
import os
import requests
from datetime import datetime
from typing import List, Optional, Dict, Any
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok

try:
    from twilio.rest import Client as TwilioClient
except ImportError:
    TwilioClient = None

# ------------------------------------------------------------------------
# Step 2: Load your real 289-ward combined risk file
# BEFORE running this cell: upload delhi_ward_combined_risk_final.csv to
# this Colab session (folder icon -> upload, or:
#   from google.colab import files
#   files.upload()
# Expected columns: ward_id, ward_name, area_km2, population_2011,
# population_density, vulnerability_weight, data_source, satellite_LST_C
# ------------------------------------------------------------------------
WARD_DATA_FILE = "delhi_ward_combined_risk_final.csv"
CANONICAL_WARDS = {}      # ward_id -> ward_name
WARD_VULNERABILITY = {}   # ward_id -> vulnerability_weight (float)
WARD_LST = {}             # ward_id -> satellite_LST_C (float)

def load_ward_data(path):
    if not os.path.exists(path):
        print(f"⚠️  {path} not found. Upload it to this Colab session, "
              f"then re-run this cell. Falling back to an empty ward list for now.")
        return
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            w_id = row["ward_id"]
            CANONICAL_WARDS[w_id] = row["ward_name"]
            try:
                WARD_VULNERABILITY[w_id] = float(row["vulnerability_weight"])
            except (KeyError, ValueError):
                WARD_VULNERABILITY[w_id] = 1.0
            try:
                WARD_LST[w_id] = float(row["satellite_LST_C"])
            except (KeyError, ValueError):
                WARD_LST[w_id] = None

load_ward_data(WARD_DATA_FILE)
print(f"✅ Loaded {len(CANONICAL_WARDS)} wards with vulnerability + satellite data")

HIGH_VULNERABILITY_THRESHOLD = 1.5  # your vulnerability_weight range is ~1.0-2.0

# ------------------------------------------------------------------------
# Step 3: Hospital directory (OUR side - real named hospitals, not ML's
# generic "Center N" placeholders). NOTE: phone numbers are PLACEHOLDERS -
# replace with real nodal officer numbers before the actual demo. Zone
# matching is prefix-based (CANT_, NDMC_) with everything else falling
# back to the default hospital.
# ------------------------------------------------------------------------
ZONE_HOSPITAL_DB = {
    "CANT": {
        "hospital_name": "Base Hospital, Delhi Cantonment",
        "contacts": {"hospital_nodal": "PLACEHOLDER"},
    },
    "NDMC": {
        "hospital_name": "Dr. Ram Manohar Lohia (RML) Hospital",
        "contacts": {"hospital_nodal": "PLACEHOLDER"},
    },
}
DEFAULT_HOSPITAL = {
    "hospital_name": "LNJP / Safdarjung Central Hospital",
    "contacts": {"hospital_nodal": "PLACEHOLDER"},
}

def get_hospital_for_ward(ward_id: str):
    for prefix, info in ZONE_HOSPITAL_DB.items():
        if ward_id.startswith(prefix):
            return info
    return DEFAULT_HOSPITAL

# ------------------------------------------------------------------------
# OPTIONAL: real cooling-center names, once available. ML currently only
# gives generic labels like "Center 1".."Center 6" (hardcoded on their
# side). Fill this in when a real directory exists - if a label isn't
# found here, we just pass ML's generic label through unchanged.
# Example: REAL_CENTER_NAMES = {"Center 1": "Community Hall, Okhla Zone"}
# ------------------------------------------------------------------------
REAL_CENTER_NAMES: Dict[str, str] = {}

def resolve_center_names(center_labels: List[str]) -> str:
    resolved = [REAL_CENTER_NAMES.get(c, c) for c in center_labels]
    return ", ".join(resolved) if resolved else "nearest designated center"

# ------------------------------------------------------------------------
# Step 4: Recipients for the three audience tiers.
# ADMIN = city administration / disaster management cell.
# PUBLIC = general residents. WORKERS = outdoor-labor-specific recipients
# (gets the same public message, which already carries labor_advisory
# content for High Risk/Extreme). In the Twilio Sandbox, each number must
# first send "join <your-sandbox-code>" to the sandbox WhatsApp number once.
# Replace these placeholders with real numbers before your demo.
# ------------------------------------------------------------------------
ADMIN_CONTACTS = ["+918800805704"]
GENERAL_PUBLIC_CONTACTS = ["+918800805704", "+918800805704"]
WORKER_CONTACTS = ["+918800805704"]
ELDERLY_CONTACTS = ["+918800805704"]
PUBLIC_RECIPIENTS = GENERAL_PUBLIC_CONTACTS + WORKER_CONTACTS + ELDERLY_CONTACTS

# ------------------------------------------------------------------------
# Step 5: Twilio WhatsApp - REAL sending, with a labelled simulation
# fallback if not configured yet.
# 1. Sign up free at twilio.com, verify your phone.
# 2. Console -> Messaging -> Try it out -> Send a WhatsApp message, to
#    activate your free Sandbox and get your Account SID + Auth Token.
# 3. Each recipient number (including your own, for testing) must WhatsApp
#    the sandbox number with the "join ..." code Twilio gives you, once.
# 4. Content Template SID (HX...) from your Content Template Builder
#    single-variable template ({{1}}) goes below too.
# ------------------------------------------------------------------------
TWILIO_ACCOUNT_SID = "AC0d097e9bc715993ed8640cb8ac56bc64"
TWILIO_AUTH_TOKEN = "7d8dcd5cfd81e4432bf00e5a7bdc0c37"
TWILIO_WHATSAPP_FROM = "whatsapp:+14155238886"  # Twilio's standard sandbox number
TWILIO_CONTENT_SID = "HX2bdefa3f0f11fcf6741bb65fb26661e7"  # starts with HX...

_twilio_client = None
_TWILIO_CONFIGURED = (
    TwilioClient is not None
    and TWILIO_ACCOUNT_SID != "AC0d097e9bc715993ed8640cb8ac56bc64"
    and TWILIO_AUTH_TOKEN != "7d8dcd5cfd81e4432bf00e5a7bdc0c37"
)
if _TWILIO_CONFIGURED:
    _twilio_client = TwilioClient(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
    print("✅ Twilio WhatsApp configured - alerts will be sent for real.")
else:
    print("⚠️  Twilio not configured - alerts will be SIMULATED (printed only) "
          "until you add your Account SID + Auth Token above.")

def send_whatsapp_message(to_number: str, body: str):
    """Sends a real WhatsApp message via Twilio (using the Content Template
    if configured, otherwise a plain-body message for Sandbox testing before
    the template is approved), or clearly prints a simulated version so
    nothing silently fails."""
    if to_number.startswith("PLACEHOLDER"):
        print(f"⚠️  [SKIPPED - placeholder number] Would send to {to_number}: \"{body}\"")
        return "SKIPPED_PLACEHOLDER"
    if _TWILIO_CONFIGURED:
        try:
            kwargs = dict(from_=TWILIO_WHATSAPP_FROM, to=f"whatsapp:{to_number}")
            if TWILIO_CONTENT_SID and TWILIO_CONTENT_SID != "HX2bdefa3f0f11fcf6741bb65fb26661e7":
                kwargs["content_sid"] = TWILIO_CONTENT_SID
                kwargs["content_variables"] = f'{{"1": {body!r}}}'
            else:
                kwargs["body"] = body
            msg = _twilio_client.messages.create(**kwargs)
            print(f"🚀 [SENT via Twilio, sid={msg.sid}] To: {to_number} -> \"{body}\"")
            return "SENT"
        except Exception as e:
            print(f"❌ [SEND FAILED] To: {to_number}: {e}")
            return "FAILED"
    else:
        print(f"🟡 [SIMULATED - Twilio not configured] To: {to_number} -> \"{body}\"")
        return "SIMULATED"

# ------------------------------------------------------------------------
# Step 6: FastAPI App
# ------------------------------------------------------------------------
app = FastAPI(
    title="Heat Action Plan - Gateway Microservice (v5)",
    description="Middleware pulling ward risk/resource data from ML's single "
                 "/forecast-heatwave source of truth and dispatching real, "
                 "bilingual, segmented (hospital/admin/public/worker) WhatsApp advisories.",
)
from fastapi.middleware.cors import CORSMiddleware
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
# ------------------------------------------------------------------------
# Step 7: THREE-TIER, BILINGUAL ADVISORY TEMPLATES
# Normal is intentionally absent - no advisory needed at Normal.
# Templates now interpolate REAL numbers from ML: risk_score, beds,
# cooling center count/names, mortality_index, ambulance priority.
# ------------------------------------------------------------------------
TEMPLATES = {
    "Caution": {
        "hospital": {
            "en": "🟡 Ward {ward_name}: Caution (risk score {risk_score}). Monitor OPD heat-related cases. Ambulance priority: {ambulance_priority}.",
            "hi": "🟡 वार्ड {ward_name}: सावधानी (जोखिम स्कोर {risk_score})। ओपीडी में गर्मी से जुड़े मामलों पर नज़र रखें। एम्बुलेंस प्राथमिकता: {ambulance_priority}।",
        },
        "admin": {
            "en": "🟡 Ward {ward_name}: Caution level. Keep {centers_count} cooling center(s) ready to open: {centers_list}.",
            "hi": "🟡 वार्ड {ward_name}: सावधानी स्तर। {centers_count} कूलिंग सेंटर तैयार रखें: {centers_list}।",
        },
        "public": {
            "en": "🟡 Heat Caution for {ward_name}: Stay hydrated, avoid strenuous activity in the afternoon sun.",
            "hi": "🟡 {ward_name} में गर्मी की सावधानी: पानी पीते रहें, दोपहर की धूप में ज़्यादा मेहनत वाला काम न करें।",
        },
    },
    "High Risk": {
        "hospital": {
            "en": "🟧 HIGH RISK{priority_tag}: Ward {ward_name}. {hospital_name}: Prepare {beds} emergency beds. Risk score {risk_score}, mortality index {mortality_index}, ambulance priority: {ambulance_priority}.",
            "hi": "🟧 उच्च जोखिम{priority_tag}: वार्ड {ward_name}। {hospital_name}: {beds} आपातकालीन बेड तैयार करें। जोखिम स्कोर {risk_score}, मृत्यु दर सूचकांक {mortality_index}, एम्बुलेंस प्राथमिकता: {ambulance_priority}।",
        },
        "admin": {
            "en": "🟧 HIGH RISK{priority_tag}: Ward {ward_name}. Open {centers_count} cooling centers now ({centers_list}) and shift outdoor work hours to avoid 12PM-4PM. {labor_advisory}",
            "hi": "🟧 उच्च जोखिम{priority_tag}: वार्ड {ward_name}। अभी {centers_count} कूलिंग सेंटर खोलें ({centers_list}) और बाहरी काम का समय दोपहर 12 से 4 बजे के बाहर रखें। {labor_advisory}",
        },
        "public": {
            "en": "🟧 HIGH HEAT RISK in {ward_name}: Avoid going outside between 12PM-4PM. Drink water often. Check on elderly and children. Nearest cooling center: {centers_list}.",
            "hi": "🟧 {ward_name} में गर्मी का उच्च खतरा: दोपहर 12 से 4 बजे तक बाहर न जाएं। बार-बार पानी पिएं। बुजुर्गों और बच्चों का ध्यान रखें। नज़दीकी कूलिंग सेंटर: {centers_list}।",
        },
    },
    "Extreme": {
        "hospital": {
            "en": "🔴 CRITICAL EXTREME{priority_tag}: Ward {ward_name}! {hospital_name}: Activate full Emergency Heat Protocol - {beds} beds needed. Risk score {risk_score}, mortality index {mortality_index}. Ambulance priority: {ambulance_priority}.",
            "hi": "🔴 अति गंभीर{priority_tag}: वार्ड {ward_name}! {hospital_name}: तुरंत पूर्ण आपातकालीन हीट प्रोटोकॉल लागू करें - {beds} बेड चाहिए। जोखिम स्कोर {risk_score}, मृत्यु दर सूचकांक {mortality_index}। एम्बुलेंस प्राथमिकता: {ambulance_priority}।",
        },
        "admin": {
            "en": "🔴 CRITICAL EXTREME{priority_tag}: Ward {ward_name}! Activate full Heat Action Plan - open ALL {centers_count} cooling centers ({centers_list}), prioritize power grid load for cooling, halt outdoor work, alert disaster management cell. {labor_advisory}",
            "hi": "🔴 अति गंभीर{priority_tag}: वार्ड {ward_name}! पूर्ण हीट एक्शन प्लान लागू करें - सभी {centers_count} कूलिंग सेंटर खोलें ({centers_list}), बिजली ग्रिड को कूलिंग के लिए प्राथमिकता दें, बाहरी काम रोकें, आपदा प्रबंधन सेल को सूचित करें। {labor_advisory}",
        },
        "public": {
            "en": "🔴 EXTREME HEAT DANGER in {ward_name}: Stay indoors. Go to your nearest cooling center immediately ({centers_list}). If you see signs of heatstroke (confusion, no sweating, very high body temperature), call 108 now.",
            "hi": "🔴 {ward_name} में अत्यधिक गर्मी का खतरा: घर के अंदर रहें। तुरंत नज़दीकी कूलिंग सेंटर जाएं ({centers_list})। अगर लू लगने के लक्षण (भ्रम, पसीना न आना, बहुत तेज़ बुखार) दिखें तो तुरंत 108 पर कॉल करें।",
        },
    },
}

RISK_RANK = {"Normal": 0, "Caution": 1, "High Risk": 2, "Extreme": 3}

# In-memory state
previous_state_db = {}
latest_vis_logs = []
state_lock = threading.Lock()

def dispatch_tier(template: dict, fmt_kwargs: dict, recipients: list):
    """Formats and sends one tier's bilingual message to its recipient list."""
    msg_en = template["en"].format(**fmt_kwargs)
    msg_hi = template["hi"].format(**fmt_kwargs)
    combined = f"{msg_en}\n\n{msg_hi}"
    for number in recipients:
        send_whatsapp_message(number, combined)
    return {"en": msg_en, "hi": msg_hi}

def process_ward_result(item: Dict[str, Any]) -> dict:
    """Shared logic: takes one ward's data straight from ML's
    /forecast-heatwave response, decides whether to alert, and if so
    dispatches all tiers (hospital/admin/public+worker), bilingual."""
    w_id = item.get("ward_id")
    w_name = item.get("ward_name") or CANONICAL_WARDS.get(w_id, w_id)
    curr_risk = item.get("risk_tier", "Normal")
    risk_score = item.get("risk_score")
    mortality_index = item.get("mortality_index")

    resources = item.get("resource_estimates", {}) or {}
    beds = resources.get("required_heat_stroke_beds")
    centers_count = resources.get("cooling_centers_count")
    ambulance_priority = resources.get("ambulance_dispatch_priority")

    actions = item.get("action_recommendations", {}) or {}
    centers_list_raw = actions.get("cooling_centers_to_activate", [])
    centers_list = resolve_center_names(centers_list_raw)
    labor_advisory = actions.get("labor_advisory", "")
    ml_sms_flag = actions.get("sms_alert_dispatched")  # informational only - ML does NOT actually send

    hosp = get_hospital_for_ward(w_id)
    hosp_name = hosp["hospital_name"]
    hosp_contacts = list(hosp["contacts"].values())

    vulnerability = WARD_VULNERABILITY.get(w_id)
    is_high_vulnerability = vulnerability is not None and vulnerability >= HIGH_VULNERABILITY_THRESHOLD
    priority_tag = " [PRIORITY - HIGH VULNERABILITY WARD]" if is_high_vulnerability else ""

    prev_risk = previous_state_db.get(w_id, "Normal")
    is_escalated = RISK_RANK.get(curr_risk, 0) > RISK_RANK.get(prev_risk, 0)

    status = "NO_CHANGE"
    messages = {"hospital": None, "admin": None, "public": None}

    if is_escalated and curr_risk in TEMPLATES:
        tier_templates = TEMPLATES[curr_risk]
        fmt_kwargs = dict(
            ward_name=w_name, hospital_name=hosp_name, priority_tag=priority_tag,
            risk_score=risk_score, mortality_index=mortality_index,
            beds=beds, centers_count=centers_count, centers_list=centers_list,
            ambulance_priority=ambulance_priority, labor_advisory=labor_advisory,
        )
        messages["hospital"] = dispatch_tier(tier_templates["hospital"], fmt_kwargs, hosp_contacts)
        messages["admin"] = dispatch_tier(tier_templates["admin"], fmt_kwargs, ADMIN_CONTACTS)
        messages["public"] = dispatch_tier(tier_templates["public"], fmt_kwargs, PUBLIC_RECIPIENTS)
        status = "DISPATCHED"
        previous_state_db[w_id] = curr_risk

    return {
        "ward_id": w_id,
        "ward_name": w_name,
        "risk_tier": curr_risk,
        "risk_score": risk_score,
        "mortality_index": mortality_index,
        "required_heat_stroke_beds": beds,
        "cooling_centers_count": centers_count,
        "cooling_centers_to_activate": centers_list_raw,
        "ambulance_dispatch_priority": ambulance_priority,
        "ml_sms_flag_informational_only": ml_sms_flag,
        "vulnerability_weight": vulnerability,
        "satellite_LST_C": WARD_LST.get(w_id),
        "assigned_hospital": hosp_name,
        "alert_status": status,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "hospital_message": messages["hospital"],
        "admin_message": messages["admin"],
        "public_message": messages["public"],
    }

@app.get("/ward-status")
def get_vis_dashboard_data():
    with state_lock:
        return {"dashboard_feed": latest_vis_logs}

@app.post("/trigger-alert")
def manual_trigger(ml_payload: List[Dict[str, Any]]):
    """Manual/dev-testing entry point - accepts a list of ward result dicts
    in the SAME shape as one item from /forecast-heatwave's "data" array."""
    global latest_vis_logs
    results = [process_ward_result(item) for item in ml_payload]
    with state_lock:
        latest_vis_logs = results
    return {"message": "Data processed & alerts dispatched", "processed_count": len(ml_payload)}

# ------------------------------------------------------------------------
# Step 8: Background polling loop - pulls from ML's SINGLE authoritative
# /forecast-heatwave endpoint. ONE request per cycle now (not 289).
# ------------------------------------------------------------------------
ML_API_BASE = "https://n54g5228-8000.inc1.devtunnels.ms/api/v1/forecast-heatwave"
POLL_INTERVAL_SECONDS = 300  # every 5 minutes - lower this for a live demo
def poll_ml_team_data():
    global latest_vis_logs
    while True:
        try:
            resp = requests.get(ML_API_BASE, params={"days": 1, "temp_offset": 0}, timeout=15)
            if resp.status_code == 200:
                payload = resp.json()
                wards = payload.get("data", [])
                updated_logs = [process_ward_result(item) for item in wards]
                if updated_logs:
                    with state_lock:
                        latest_vis_logs = updated_logs
                    print(f"🔄 Polled {len(updated_logs)} wards from /forecast-heatwave at "
                          f"{datetime.now().strftime('%H:%M:%S')}")
            else:
                print(f"⚠️  Poll failed: HTTP {resp.status_code}")
        except requests.RequestException as e:
            print(f"⚠️  Poll failed: {e}")
        time.sleep(POLL_INTERVAL_SECONDS)
# ------------------------------------------------------------------------
# Step 9: Public tunnel - REMINDER: don't share this URL with the team yet.
# Only generate + share the FINAL one once everything is fully integrated.
# ------------------------------------------------------------------------
NGROK_AUTH_TOKEN = "3JBhxi174lkRvcf6CsGcPR1FooP_E7ozwUgqxuo71kC58vd"
if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "3JBhxi174lkRvcf6CsGcPR1FooP_E7ozwUgqxuo71kC58vd":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8000)
    print(f"🌍 Public URL (do NOT share yet - for your own testing only): {public_url}")
else:
    print("⚠️  No ngrok token set - this API will only be reachable inside this Colab session.")
# ------------------------------------------------------------------------
# Step 10: Run server + polling loop in background threads (non-blocking)
# ------------------------------------------------------------------------
def run_colab_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

server_thread = threading.Thread(target=run_colab_server, daemon=True)
server_thread.start()

pooling_thread = threading.Thread(target=poll_ml_team_data, daemon=True)
pooling_thread.start()

print("✅ Server running on http://127.0.0.1:8000 (and on the ngrok URL above, if set)")
print("✅ Background polling of ML's /forecast-heatwave has started")
# ------------------------------------------------------------------------
# Step 11: Quick manual test (optional) - fires all tiers for two sample
# wards (one High Risk, one Extreme) using the EXACT shape ML actually
# returns, so you can see hospital + admin + public messages end-to-end
# without waiting for the live poll cycle.
# ------------------------------------------------------------------------
time.sleep(1)
sample_payload = [
    {
        "ward_id": "CANT_1", "ward_name": "DELHI CANTT CHARGE 1",
        "wbgt": 26.08, "risk_score": 50.34,
        "risk_tier": "High Risk",
        "mortality_index": 27.62,
        "resource_estimates": {"required_heat_stroke_beds": 50, "cooling_centers_count": 3,
                               "ambulance_dispatch_priority": "High"},
        "action_recommendations": {
            "cooling_centers_to_activate": ["Center 5", "Center 6", "Center 3", "Center 1"],
            "labor_advisory": "Restrict strenuous work from 12 PM to 4 PM.",
            "healthcare_outreach": "Deploy ASHA workers to high-risk zones.",
            "sms_alert_dispatched": True,
        },
        "forecast_day": 1,
    },
    {
        "ward_id": "142", "ward_name": "RAJ NAGAR",
        "wbgt": 25.91, "risk_score": 87.31,
        "risk_tier": "Extreme",
        "mortality_index": 47.31,
        "resource_estimates": {"required_heat_stroke_beds": 150, "cooling_centers_count": 6,
                               "ambulance_dispatch_priority": "Extreme"},
        "action_recommendations": {
            "cooling_centers_to_activate": ["Center 6", "Center 5", "Center 4", "Center 2"],
            "labor_advisory": "Total restriction of outdoor work.",
            "healthcare_outreach": "Full mobilization of emergency response teams.",
            "sms_alert_dispatched": True,
        },
        "forecast_day": 1,
    },
]
try:
    r = requests.post("http://127.0.0.1:8000/trigger-alert", json=sample_payload, timeout=5)
    print("\n--- Manual test POST response ---")
    print(r.json())
    g = requests.get("http://127.0.0.1:8000/ward-status", timeout=5)
    print("\n--- Current dashboard feed (all tiers included) ---")
    print(g.json())
except requests.RequestException as e:
    print(f"Manual test skipped: {e}")

✅ Loaded 289 wards with vulnerability + satellite data
⚠️  Twilio not configured - alerts will be SIMULATED (printed only) until you add your Account SID + Auth Token above.
⚠️  No ngrok token set - this API will only be reachable inside this Colab session.
✅ Server running on http://127.0.0.1:8000 (and on the ngrok URL above, if set)
✅ Background polling of ML's /forecast-heatwave has started


ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


⚠️  [SKIPPED - placeholder number] Would send to PLACEHOLDER: "🟧 HIGH RISK: Ward DELHI CANTT CHARGE 1. Base Hospital, Delhi Cantonment: Prepare 50 emergency beds. Risk score 50.34, mortality index 27.62, ambulance priority: High.

🟧 उच्च जोखिम: वार्ड DELHI CANTT CHARGE 1। Base Hospital, Delhi Cantonment: 50 आपातकालीन बेड तैयार करें। जोखिम स्कोर 50.34, मृत्यु दर सूचकांक 27.62, एम्बुलेंस प्राथमिकता: High।"
🟡 [SIMULATED - Twilio not configured] To: +918800805704 -> "🟧 HIGH RISK: Ward DELHI CANTT CHARGE 1. Open 3 cooling centers now (Center 5, Center 6, Center 3, Center 1) and shift outdoor work hours to avoid 12PM-4PM. Restrict strenuous work from 12 PM to 4 PM.

🟧 उच्च जोखिम: वार्ड DELHI CANTT CHARGE 1। अभी 3 कूलिंग सेंटर खोलें (Center 5, Center 6, Center 3, Center 1) और बाहरी काम का समय दोपहर 12 से 4 बजे के बाहर रखें। Restrict strenuous work from 12 PM to 4 PM."
🟡 [SIMULATED - Twilio not configured] To: +918800805704 -> "🟧 HIGH HEAT RISK in DELHI CANTT CHARGE 1: Avoid going outside betw

In [ ]:
# ============================================================
# LIVE PATCH — updates Twilio + starts ngrok WITHOUT restarting
# the server or rerunning the big cell. Run this once, after
# your main v5 cell is already running.
# ============================================================
from twilio.rest import Client as TwilioClient
from pyngrok import ngrok

# ---- Twilio: overwrite the globals send_whatsapp_message() reads ----
TWILIO_ACCOUNT_SID = "AC0d097e9bc715993ed8640cb8ac56bc64"
TWILIO_AUTH_TOKEN  = "7d8dcd5cfd81e4432bf00e5a7bdc0c37"
TWILIO_CONTENT_SID = "HX2bdefa3f0f11fcf6741bb65fb26661e7"

_TWILIO_CONFIGURED = True
_twilio_client = TwilioClient(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
print("✅ Twilio patched — real sends now active.")

# ---- ngrok: open a tunnel to the ALREADY-RUNNING server on :8000 ----
NGROK_AUTH_TOKEN = "3JBhxi174lkRvcf6CsGcPR1FooP_E7ozwUgqxuo71kC58vd"
ngrok.kill()  # clear any half-open tunnel from before
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print(f"🌍 Public URL (do NOT share yet): {public_url}")

✅ Twilio patched — real sends now active.
🌍 Public URL (do NOT share yet): NgrokTunnel: "https://sandstorm-either-detail.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
TWILIO_CONTENT_SID = "HX2bdefa3f0f11fcf6741bb65fb26661e7"

In [ ]:
!pip install pyngrok fastapi uvicorn requests pydantic python-multipart nest_asyncio twilio -q

In [ ]:
previous_state_db.clear()
print("✅ Cleared previous dispatch state — next test will fire fresh alerts.")

✅ Cleared previous dispatch state — next test will fire fresh alerts.


In [ ]:
import json
test_payload = [{
    "ward_id": "MANUAL_TEST_1", "ward_name": "MANUAL TEST WARD",
    "wbgt": 25.9, "risk_score": 90.0, "risk_tier": "Extreme",
    "mortality_index": 50.0,
    "resource_estimates": {"required_heat_stroke_beds": 100, "cooling_centers_count": 4, "ambulance_dispatch_priority": "Emergency"},
    "action_recommendations": {
        "cooling_centers_to_activate": ["Center 1", "Center 2"],
        "labor_advisory": "Total restriction of outdoor work.",
        "healthcare_outreach": "Full mobilization.",
        "sms_alert_dispatched": True
    },
    "forecast_day": 1
}]
results = [process_ward_result(item) for item in test_payload]
print(json.dumps(results, indent=2, ensure_ascii=False))

⚠️  [SKIPPED - placeholder number] Would send to PLACEHOLDER: "🔴 CRITICAL EXTREME: Ward MANUAL TEST WARD! LNJP / Safdarjung Central Hospital: Activate full Emergency Heat Protocol - 100 beds needed. Risk score 90.0, mortality index 50.0. Ambulance priority: Emergency.

🔴 अति गंभीर: वार्ड MANUAL TEST WARD! LNJP / Safdarjung Central Hospital: तुरंत पूर्ण आपातकालीन हीट प्रोटोकॉल लागू करें - 100 बेड चाहिए। जोखिम स्कोर 90.0, मृत्यु दर सूचकांक 50.0। एम्बुलेंस प्राथमिकता: Emergency।"
❌ [SEND FAILED] To: +918800805704: HTTP 400 error: Unable to create record: ContentSid Required
❌ [SEND FAILED] To: +918800805704: HTTP 400 error: Unable to create record: ContentSid Required
❌ [SEND FAILED] To: +918800805704: HTTP 400 error: Unable to create record: ContentSid Required
❌ [SEND FAILED] To: +918800805704: HTTP 400 error: Unable to create record: ContentSid Required
❌ [SEND FAILED] To: +918800805704: HTTP 400 error: Unable to create record: ContentSid Required
[
  {
    "ward_id": "MANUAL_TEST_1",


In [ ]:
try:
    msg = _twilio_client.messages.create(
        from_=TWILIO_WHATSAPP_FROM,
        to="whatsapp:+918800805704",
        content_sid=TWILIO_CONTENT_SID,
        content_variables='{"1": "Test message from gateway"}'
    )
    print(f"✅ SUCCESS: {msg.sid}")
except Exception as e:
    print(f"❌ FULL ERROR:\n{e}")